In [14]:
import torch
import json
import os
os.environ["HF_ENDPOINT"]="https://hf-mirror.com"
os.environ["CUDA_VISIBLE_DEVICES"]="7"
from datasets import load_dataset

In [15]:
huatuo = load_dataset("json", data_files="/nvme/zzlai/MedicalGPT/mydata/HuatuoGPT2-GPT4-SFT-140K.jsonl", split="train")

In [5]:
huatuo = load_dataset("json", data_files="/nvme/zzlai/hf_cache/hub/datasets--FreedomIntelligence--HuatuoGPT2-SFT-GPT4-140K/snapshots/4077ffaeb123e49b8b8a0283f42957a5570a52ce/HuatuoGPT2-GPT4-SFT-140K.json", split="train")

In [ ]:
huatuo['train'][0]

{'id': 'huatuogpt2_sft_1',
 'conversations': [{'from': 'human',
   'value': '我想咨询一下，在月经来的什么时候做爱最安全，还有刚做完药流，2个人最好得什么时候可以在一去，如果当天就在一起了会不会有什么不好呢第一次问题补充:我想咨询一下，在月经来前的什么时候做爱最安全，还有刚做完药流，2个人得多长时间可以在一起会比较好，如果隔几天就在一起了会不会有什么不好后果呢'},
  {'from': 'gpt',
   'value': '关于你的问题，我会分别进行解答。\n\n1. 在月经周期中，哪个时期性行为最安全？\n   月经周期通常被分为几个阶段：月经期、排卵期、安全期。安全期是指女性月经周期中不易受孕的时间段，通常在月经结束后的前几天和月经来潮前的几天。然而，这并不是一个绝对的规则，因为每个女性的月经周期可能会有所不同，而且排卵期也可能会因为各种原因（如压力、疾病等）而改变。因此，如果你想避免怀孕，最安全的方法是使用避孕措施，如避孕套或避孕药。\n\n2. 药流后多久可以进行性行为？\n   药流后，子宫内膜需要一段时间来恢复。通常，医生会建议在药流后至少等待2-4周再进行性行为，以防止感染和出血。在这段时间内，你可能会经历一些出血或分泌物，这是正常的。如果出血过多或持续时间过长，应立即就医。\n\n3. 药流后立即进行性行为会有什么后果？\n   药流后立即进行性行为可能会增加感染的风险，因为子宫颈可能还没有完全关闭，这可能会使细菌更容易进入子宫。此外，它也可能导致出血加重或持续时间延长。因此，最好在药流后等待一段时间再进行性行为。\n\n总的来说，无论是在月经周期中的哪个阶段，还是在药流后，都应该使用避孕措施来避免不期望的怀孕。如果你有任何疑问或担忧，建议你咨询医生或其他医疗专业人士。'}]}

In [40]:
import json

path = "/nvme/zzlai/MedicalGPT/mydata/sharegpt_zh_38K.jsonl"

bad = []
with open(path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)

        convs = obj.get("conversations", [])
        if not isinstance(convs, list):
            bad.append((line_no, "conversations_not_list"))
            continue

        for turn_id, msg in enumerate(convs):
            if not isinstance(msg, dict):
                bad.append((line_no, turn_id, "msg_not_dict"))
                continue

            extra_keys = set(msg.keys()) - {"from", "value"}
            if extra_keys:
                bad.append((line_no, turn_id, sorted(extra_keys)))

print("bad count =", len(bad))
print("first 20 =", bad[:20])

bad count = 8
first 20 = [(16472, 1, ['markdown', 'text']), (16472, 3, ['markdown', 'text']), (16472, 5, ['markdown', 'text']), (16472, 7, ['markdown', 'text']), (16472, 9, ['markdown', 'text']), (16472, 11, ['markdown', 'text']), (16472, 13, ['markdown', 'text']), (24758, 1, ['markdown', 'text'])]


In [42]:
import json

src = "/nvme/zzlai/MedicalGPT/mydata/sharegpt_zh_38K.jsonl"
dst = "/nvme/zzlai/MedicalGPT/mydata/sharegpt_zh_38K_clean.jsonl"

num_total = 0
num_kept = 0
num_bad_json = 0
num_bad_sample = 0
num_bad_turn = 0

with open(src, "r", encoding="utf-8") as fin, open(dst, "w", encoding="utf-8") as fout:
    for line_no, line in enumerate(fin, 1):
        line = line.strip()
        if not line:
            continue
        num_total += 1

        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            num_bad_json += 1
            continue

        convs = obj.get("conversations")
        if not isinstance(convs, list) or len(convs) == 0:
            num_bad_sample += 1
            continue

        new_convs = []
        valid = True

        for msg in convs:
            if not isinstance(msg, dict):
                valid = False
                break

            role = msg.get("from")
            value = msg.get("value")

            if not isinstance(role, str) or not isinstance(value, str):
                valid = False
                break

            # 关键：重新构造，只保留 from/value
            new_convs.append({
                "from": role,
                "value": value,
            })

        if not valid or len(new_convs) == 0:
            num_bad_turn += 1
            continue

        clean_obj = {
            "id": str(obj.get("id", line_no)),
            "conversations": new_convs,
            "lang": obj.get("lang", "") if isinstance(obj.get("lang", ""), str) else "",
        }

        fout.write(json.dumps(clean_obj, ensure_ascii=False) + "\n")
        num_kept += 1

print("total     =", num_total)
print("kept      =", num_kept)
print("bad_json  =", num_bad_json)
print("bad_sample=", num_bad_sample)
print("bad_turn  =", num_bad_turn)
print("saved to  =", dst)

total     = 38557
kept      = 38557
bad_json  = 0
bad_sample= 0
bad_turn  = 0
saved to  = /nvme/zzlai/MedicalGPT/mydata/sharegpt_zh_38K_clean.jsonl


In [43]:
import json

clean_path = "/nvme/zzlai/MedicalGPT/mydata/sharegpt_zh_38K_clean.jsonl"

with open(clean_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        for j, msg in enumerate(obj["conversations"]):
            extra = set(msg.keys()) - {"from", "value"}
            if extra:
                print("still dirty:", i, j, extra)
                raise SystemExit
print("all clean")

all clean


In [44]:
huatuo = load_dataset("json", data_files="/nvme/zzlai/MedicalGPT/mydata/sharegpt_zh_38K_clean.jsonl", split="train")

Generating train split: 38557 examples [00:01, 23598.72 examples/s]


In [2]:
from sentence_transformers import SentenceTransformer

# Load the model
# model = SentenceTransformer("Qwen/Qwen3-Embedding-8B")

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B",
    model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "cuda"},
    tokenizer_kwargs={"padding_side": "left"},
)

# The queries and documents to embed
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

# Encode the queries and documents. Note that queries benefit from using a prompt
# Here we use the prompt called "query" stored under `model.prompts`, but you can
# also pass your own prompt via the `prompt` argument
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)
# tensor([[0.7493, 0.0751],
#         [0.0880, 0.6318]])


The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
Loading weights: 100%|██████████| 398/398 [00:04<00:00, 98.20it/s] 


tensor([[0.7466, 0.0745],
        [0.0888, 0.6308]])
